# Decision Consistency Evaluation — CIFAR-10 / CIFAR-100

Measures Label Stability (S), Confidence Deviation (D), and Consistency Score (C = alpha*S + (1-alpha)*(1-D)) across 4 ImageNet-pretrained CNNs.

Six non-adversarial perturbations: rotation, brightness, noise, contrast, blur, JPEG.

**Update paths in Configuration cell before running.**

In [ ]:
CIFAR10_DIR  = r'E:\cifar-10-python\cifar-10-batches-py'
CIFAR100_DIR = r'E:\cifar-100-python\cifar-100-python'
OUTPUT_DIR   = r'E:\decision_consistency_cifar_outputs'
NUM_SAMPLES  = 1000
SEED         = 42
ALPHA        = 0.5

In [ ]:
import os, pickle, json, random, io, csv
from collections import defaultdict
import numpy as np
from scipy.stats import wilcoxon, pearsonr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

for sub in ['tables','figures/consistency','figures/gradcam','figures/perclass']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
def load_cifar10(d):
    with open(os.path.join(d,'test_batch'),'rb') as f: b=pickle.load(f,encoding='bytes')
    return b[b'data'].reshape(-1,3,32,32), np.array(b[b'labels']), ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

X10,y10,cls10 = load_cifar10(CIFAR10_DIR)
DATASETS = {'CIFAR-10':(X10,y10,cls10)}
if os.path.isdir(CIFAR100_DIR):
    def load_cifar100(d):
        with open(os.path.join(d,'test'),'rb') as f: b=pickle.load(f,encoding='bytes')
        with open(os.path.join(d,'meta'),'rb') as f: m=pickle.load(f,encoding='bytes')
        return b[b'data'].reshape(-1,3,32,32), np.array(b[b'fine_labels']), [n.decode() for n in m[b'fine_label_names']]
    X100,y100,cls100 = load_cifar100(CIFAR100_DIR)
    DATASETS['CIFAR-100'] = (X100,y100,cls100)
    print(f'CIFAR-100: {X100.shape}')
print(f'CIFAR-10 test: {X10.shape}')

In [ ]:
zoo = [('ResNet-18',models.resnet18),('ResNet-50',models.resnet50),('VGG-16',models.vgg16),('MobileNetV2',models.mobilenet_v2)]
MODELS = {}
for name, fn in zoo:
    print(f'Loading {name}...', end=' ', flush=True)
    MODELS[name] = fn(pretrained=True).eval().to(device); print('OK')
print('Models ready:', list(MODELS.keys()))

In [ ]:
preprocess = transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])

def prepare(t): return preprocess(transforms.ToPILImage()(t.cpu().clamp(0,1))).unsqueeze(0).to(device)
def apply_jpeg(t,q=75):
    buf=io.BytesIO(); transforms.ToPILImage()(t.cpu().clamp(0,1)).save(buf,format='JPEG',quality=q); buf.seek(0); return transforms.ToTensor()(Image.open(buf))
def generate_perturbations(t):
    noise=0.01*torch.randn_like(t)
    return {'rotation':TF.rotate(t,angle=2),'brightness':TF.adjust_brightness(t,brightness_factor=1.1),'noise':torch.clamp(t+noise,0,1),'contrast':TF.adjust_contrast(t,contrast_factor=1.1),'blur':TF.gaussian_blur(t,kernel_size=3,sigma=0.5),'jpeg':apply_jpeg(t)}
def infer(model,t):
    with torch.no_grad(): probs=F.softmax(model(prepare(t)),dim=1)
    return probs.argmax(1).item(), probs.max().item()
def compute_metrics(op,oc,pr,alpha=ALPHA):
    K=len(pr); S=sum(1 for p,_ in pr.values() if p==op)/K; D=sum(abs(c-oc)/max(oc,1-oc) for _,c in pr.values())/K
    return {'Label_Stability':S,'Confidence_Deviation':D,'Consistency_Score':alpha*S+(1-alpha)*(1-D)}
print('Pipeline ready.')

In [ ]:
results = {}
for ds_name,(X,y,class_names) in DATASETS.items():
    results[ds_name]={}; X_t=torch.tensor(X,dtype=torch.float32)/255.0; n=min(NUM_SAMPLES,len(y))
    for model_name,model in MODELS.items():
        per_sample=[]
        for idx in tqdm(range(n),desc=f'{ds_name}/{model_name}'):
            img=X_t[idx]; op,oc=infer(model,img); pr={k:infer(model,v) for k,v in generate_perturbations(img).items()}
            m=compute_metrics(op,oc,pr); m['true_label']=int(y[idx]); m['class_name']=class_names[int(y[idx])]; m['orig_pred']=op; m['orig_conf']=oc
            per_sample.append(m)
        results[ds_name][model_name]=per_sample
        ls=np.mean([m['Label_Stability'] for m in per_sample]); cd=np.mean([m['Confidence_Deviation'] for m in per_sample]); cs=np.mean([m['Consistency_Score'] for m in per_sample])
        print(f'  [{ds_name}][{model_name}]  LS={ls:.3f}  CD={cd:.3f}  CS={cs:.3f}')
print('Done.')

In [ ]:
header=['Dataset','Model','Avg_Label_Stability','Avg_Confidence_Deviation','Avg_Consistency_Score','Std_Consistency_Score']
rows=[]
for ds_name in results:
    for mn in results[ds_name]:
        ms=results[ds_name][mn]; ls=np.mean([m['Label_Stability'] for m in ms]); cd=np.mean([m['Confidence_Deviation'] for m in ms]); cs=np.mean([m['Consistency_Score'] for m in ms]); std=np.std([m['Consistency_Score'] for m in ms])
        rows.append({'Dataset':ds_name,'Model':mn,'Avg_Label_Stability':round(ls,4),'Avg_Confidence_Deviation':round(cd,4),'Avg_Consistency_Score':round(cs,4),'Std_Consistency_Score':round(std,4)})
        print(f'{ds_name:<12} {mn:<14} LS={ls:.3f} CD={cd:.3f} CS={cs:.3f} std={std:.3f}')
csv_path=os.path.join(OUTPUT_DIR,'tables','summary_table.csv')
with open(csv_path,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=header); w.writeheader(); w.writerows(rows)
print('Saved:',csv_path)

In [ ]:
for ds_name in results:
    mn_list=list(results[ds_name].keys()); cs_arrays={mn:np.array([m['Consistency_Score'] for m in results[ds_name][mn]]) for mn in mn_list}
    print(f'\n=== Wilcoxon Tests ({ds_name}) ===')
    for i,mn1 in enumerate(mn_list):
        for mn2 in mn_list[i+1:]:
            w,p=wilcoxon(cs_arrays[mn1],cs_arrays[mn2]); print(f'  {mn1} vs {mn2}: W={w:.1f} p={p:.4f} {"***" if p<0.05 else "ns"}')

In [ ]:
alpha_values=[0.3,0.4,0.5,0.6,0.7]
for ds_name in results:
    fig,ax=plt.subplots(figsize=(8,4))
    for mn,marker in zip(results[ds_name].keys(),['o','s','^','D']):
        ms=results[ds_name][mn]; ls_a=np.array([m['Label_Stability'] for m in ms]); cd_a=np.array([m['Confidence_Deviation'] for m in ms])
        vals=[float(np.mean(a*ls_a+(1-a)*(1-cd_a))) for a in alpha_values]
        ax.plot(alpha_values,vals,marker=marker,label=mn,linewidth=2,markersize=7)
    ax.set_xlabel('alpha'); ax.set_ylabel('Avg Consistency Score'); ax.set_title(f'Alpha Ablation {ds_name}'); ax.set_xticks(alpha_values); ax.set_ylim(0,1); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
    path=os.path.join(OUTPUT_DIR,'figures','consistency',f'alpha_ablation_{ds_name.replace("-","_")}.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
for ds_name in results:
    mn_list=list(results[ds_name].keys())
    avg_ls=[np.mean([m['Label_Stability'] for m in results[ds_name][mn]]) for mn in mn_list]
    avg_cs=[np.mean([m['Consistency_Score'] for m in results[ds_name][mn]]) for mn in mn_list]
    avg_cd=[np.mean([m['Confidence_Deviation'] for m in results[ds_name][mn]]) for mn in mn_list]
    x,w=np.arange(len(mn_list)),0.25
    fig,ax=plt.subplots(figsize=(9,5))
    ax.bar(x-w,avg_ls,w,label='Label Stability',color='steelblue'); ax.bar(x,avg_cs,w,label='Consistency Score',color='darkorange'); ax.bar(x+w,avg_cd,w,label='Confidence Deviation',color='seagreen')
    ax.set_xticks(x); ax.set_xticklabels(mn_list,fontsize=11); ax.set_ylim(0,1); ax.set_ylabel('Score'); ax.set_title(f'Cross-Model Consistency {ds_name}',fontsize=12); ax.legend(); ax.grid(axis='y',alpha=0.3); plt.tight_layout()
    path=os.path.join(OUTPUT_DIR,'figures','consistency',f'crossmodel_{ds_name.replace("-","_")}.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
class GradCAM:
    def __init__(self,model,model_name):
        self.model=model; self.gradients=self.activations=None
        target=model.layer4[-1] if 'ResNet' in model_name else model.features[-1]
        target.register_forward_hook(lambda m,i,o: setattr(self,'activations',o.detach()))
        target.register_full_backward_hook(lambda m,gi,go: setattr(self,'gradients',go[0].detach()))
    def generate(self,inp,class_idx):
        self.model.zero_grad(); self.model(inp)[0,class_idx].backward()
        w=self.gradients.mean(dim=[2,3],keepdim=True); cam=F.relu((w*self.activations).sum(dim=1).squeeze()).cpu().numpy()
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((224,224),Image.BILINEAR))/255.0
def denorm(t):
    arr=t.squeeze().permute(1,2,0).detach().cpu().numpy()
    return np.clip(arr*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406]),0,1)
X_t=torch.tensor(X10,dtype=torch.float32)/255.0
for model_name,model in MODELS.items():
    bidx=next((i for i,m in enumerate(results['CIFAR-10'][model_name]) if m['Label_Stability']<1.0),None)
    if bidx is None: continue
    img_o=X_t[bidx]; img_p=generate_perturbations(img_o)['rotation']
    gcam=GradCAM(model,model_name); inp_o=prepare(img_o).requires_grad_(True); inp_p=prepare(img_p).requires_grad_(True)
    op,_=infer(model,img_o); pp,_=infer(model,img_p)
    cam_o=gcam.generate(inp_o,op); cam_p=gcam.generate(inp_p,pp)
    fig,axes=plt.subplots(1,2,figsize=(6,3))
    for ax,inp_t,cam,title in [(axes[0],inp_o,cam_o,f'Original pred:{op}'),(axes[1],inp_p,cam_p,f'Rotation pred:{pp}')]:
        ax.imshow(denorm(inp_t)); ax.imshow(cam,cmap='jet',alpha=0.45); ax.set_title(title,fontsize=9); ax.axis('off')
    fig.suptitle(f'Grad-CAM {model_name}/CIFAR-10',fontsize=10); plt.tight_layout()
    fname=f'gradcam_{model_name}.png'.replace('-','_'); plt.savefig(os.path.join(OUTPUT_DIR,'figures','gradcam',fname),dpi=300,bbox_inches='tight'); plt.close(); print('Saved GradCAM:',model_name)